# 07. Real-Time Vision: Webcam & Video Stream Sudoku Augmented Reality

### Overview
This notebook prototypes an interactive video pipeline for live Augmented Reality Sudoku solving:
- **State Machine**:
  - `SEARCHING`: Scanning incoming video frames for a valid 9×9 Sudoku grid.
  - `GRID_LOCKED`: When a grid is detected with high stability, lock corners and perform CNN inference.
  - `SOLVED`: Overlay green solution digits onto the live camera stream in real time.
- **Movement Thresholding**: Prevents jitter by tracking corner displacement across frames.

In [ ]:
import os
import sys
import time
from pathlib import Path

def find_root() -> Path:
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / "models" / "digit_classifier.keras").exists():
            return parent
    return Path(os.path.abspath("..")).resolve()

PROJECT_ROOT = find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np

from backend.cv_pipeline import (
    find_grid_corners,
    warp_grid,
    split_cells,
    overlay_solution,
    unwarp_overlay
)
from backend.model_service import SudokuDigitRecognizer
from backend.solver_service import solve_sudoku

print("Real-time pipeline modules ready.")

## 1. State Machine & Configuration

In [ ]:
# Video Stream Parameters
CAMERA_INDEX = 0
DETECT_EVERY_N_FRAMES = 8
MOVE_THRESHOLD = 0.04
MAX_FAILED_FRAMES = 6
WARP_SIZE = 900

# States
STATE_SEARCHING = "SEARCHING"
STATE_LOCKED = "LOCKED"
STATE_SOLVED = "SOLVED"

def grid_moved(old_corners, new_corners, frame_shape, threshold=MOVE_THRESHOLD):
    """Calculates normalized corner displacement to detect camera motion."""
    if old_corners is None or new_corners is None:
        return True
    old_c = np.asarray(old_corners, dtype=np.float32)
    new_c = np.asarray(new_corners, dtype=np.float32)
    
    movement = np.mean(np.linalg.norm(old_c - new_c, axis=1))
    h, w = frame_shape[:2]
    diagonal = np.sqrt(h**2 + w**2)
    return (movement / diagonal) > threshold

## 2. Real-Time Augmented Reality Projection

In [ ]:
def draw_realtime_solution(frame, corners, original_grid, solved_grid):
    """Overlays the solved digits onto the live video frame."""
    if corners is None or original_grid is None or solved_grid is None:
        return frame
        
    warped, M = warp_grid(frame, corners, size=WARP_SIZE)
    warped_sol = overlay_solution(warped, original_grid, solved_grid)
    augmented = unwarp_overlay(frame, warped_sol, M)
    return augmented

## 3. Video Capture Loop
Run this cell to start live webcam capture. Press **`q`** to quit.

In [ ]:
def run_video_solver():
    recognizer = SudokuDigitRecognizer(str(PROJECT_ROOT / "models" / "digit_classifier.keras"))
    
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print(f"Could not access camera index {CAMERA_INDEX}. Verify webcam connection.")
        return

    state = STATE_SEARCHING
    frame_count = 0
    failed_detections = 0
    locked_corners = None
    original_grid = None
    solved_grid = None
    
    print("Live Sudoku Solver started. Press 'q' in the window to quit.")

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_count += 1
            display = frame.copy()

            if state == STATE_SEARCHING:
                cv2.putText(display, "Searching for Sudoku Grid...", (20, 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 165, 255), 2)
                            
                if frame_count % DETECT_EVERY_N_FRAMES == 0:
                    corners = find_grid_corners(frame)
                    if corners is not None:
                        locked_corners = corners
                        state = STATE_LOCKED

            elif state == STATE_LOCKED:
                cv2.putText(display, "Grid Locked - Recognizing Digits...", (20, 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
                            
                warped, M = warp_grid(frame, locked_corners, size=WARP_SIZE)
                warped_gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
                cells = split_cells(warped_gray)
                
                original_grid, _, _ = recognizer.recognize_sudoku(cells)
                success, solved, _, _, _ = solve_sudoku(original_grid)
                
                if success:
                    solved_grid = solved
                    state = STATE_SOLVED
                else:
                    state = STATE_SEARCHING

            elif state == STATE_SOLVED:
                display = draw_realtime_solution(frame, locked_corners, original_grid, solved_grid)
                cv2.putText(display, "SOLVED (AR Active)", (20, 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                            
                # Check if grid moved
                new_corners = find_grid_corners(frame)
                if new_corners is None or grid_moved(locked_corners, new_corners, frame.shape):
                    failed_detections += 1
                    if failed_detections > MAX_FAILED_FRAMES:
                        state = STATE_SEARCHING
                        failed_detections = 0
                else:
                    locked_corners = new_corners
                    failed_detections = 0

            cv2.imshow("Sudoku AI AR Solver (Press 'q' to Quit)", display)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("Video capture released cleanly.")

# To run webcam, uncomment the line below:
# run_video_solver()